In [ ]:
FIGURE = 'Fig.S20'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
import numpy as np

def window_maps(field, pathway, fdr=False):
    import netCDF4 as nc
    import matplotlib.pyplot as plt
    from matplotlib.colors import BoundaryNorm, ListedColormap
    import cartopy.crs as ccrs
    if fdr:
        source = INPUT_DIR / 'correlation_significance.nc'
    else:
        source = INPUT_DIR / 'correlation_significance.nc'
    with open_nc(str(source)) as d:
        value = np.asarray(np.ma.filled(d[field][:].astype(float), np.nan))
        lat = np.asarray(d['lat'][:])
        lon = np.asarray(d['lon'][:])
        years = np.asarray(d['window_start_year'][:])
        if fdr:
            value = np.where(d['q_scale_at_opt'][:] < 0.05, value, np.nan)
    fig, axes = plt.subplots(1, len(years), squeeze=False, figsize=(27, 8), subplot_kw={'projection': ccrs.PlateCarree()})
    if field == 'opt_scale':
        cmap = ListedColormap(['#0476E7', '#00B2CB', '#00AA09', '#E0B200', '#FA7F00', '#E3242B'])
        norm = BoundaryNorm([0.5, 1.5, 3.5, 6.5, 12.5, 18.5, 24.5], 6)
    else:
        cmap = 'RdBu_r'
        norm = plt.Normalize(-1, 1)
    for s, season in enumerate(['MAM']):
        for w, year in enumerate(years):
            ax = axes[s, w]
            im = ax.pcolormesh(lon, lat, value[w, s], cmap=cmap, norm=norm, transform=ccrs.PlateCarree(), rasterized=True)
            ax.coastlines(linewidth=0.3)
            ax.set_extent([-180, 180, -60, 85])
            ax.set_title(f'{year}–{year + 9} {season}', fontsize=7)
    fig.colorbar(im, ax=axes.ravel().tolist(), orientation='horizontal', fraction=0.03, pad=0.04)
    out = OUTPUT / 'window_maps'
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / (f'{pathway}_{field}' + ('_FDR' if fdr else '') + '.png'), dpi=600, bbox_inches='tight')
    plt.close(fig)
window_maps('max_r', 'SSMIrz', fdr=True)
